In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision import models, datasets
from torch.utils.data import DataLoader, WeightedRandomSampler
import numpy as np
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")

✅ Using device: cuda


In [2]:
for root, dirs, files in os.walk('/kaggle/input'):
    if len(dirs) > 10:
        print(f"Found {len(dirs)} folders at: {root}")
        break
    else:
        print(root, dirs)

/kaggle/input ['datasets']
/kaggle/input/datasets ['abdallahalidev']
/kaggle/input/datasets/abdallahalidev ['plantvillage-dataset']
/kaggle/input/datasets/abdallahalidev/plantvillage-dataset ['segmented', 'grayscale', 'plantvillage dataset', 'color']
Found 38 folders at: /kaggle/input/datasets/abdallahalidev/plantvillage-dataset/segmented


In [3]:
DATA_DIR = "/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color"

In [4]:
from torch.utils.data import WeightedRandomSampler
import numpy as np

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

DATA_DIR = "/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color"

full_dataset = datasets.ImageFolder(DATA_DIR, transform=transform_train)
class_names = full_dataset.classes
num_classes = len(class_names)
print(f"✅ Classes: {num_classes}")
print(f"✅ Total images: {len(full_dataset)}")

val_size = int(0.2 * len(full_dataset))
train_size = len(full_dataset) - val_size
train_dataset, val_dataset = torch.utils.data.random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# Fix class imbalance
targets = [full_dataset.targets[i] for i in train_dataset.indices]
class_counts = np.bincount(targets)
class_weights = 1.0 / class_counts
sample_weights = [class_weights[t] for t in targets]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler, num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2)

# Model
model = models.mobilenet_v2(weights='MobileNet_V2_Weights.IMAGENET1K_V1')
model.classifier[1] = nn.Linear(model.last_channel, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

# Train 10 epochs
for epoch in range(10):
    model.train()
    running_loss, correct, total = 0, 0, 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    scheduler.step()

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    print(f"Epoch {epoch+1}/10 | Train Acc: {correct/total*100:.2f}% | Val Acc: {val_correct/val_total*100:.2f}%")

print("✅ Training complete!")
import json

# Save to Kaggle's persistent output folder
torch.save(model.state_dict(), '/kaggle/working/plant_disease_model.pth')

with open('/kaggle/working/class_names.json', 'w') as f:
    json.dump(class_names, f)

print("✅ Model saved to Kaggle output!")

✅ Classes: 38
✅ Total images: 54305
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 105MB/s] 


Epoch 1/10 | Train Acc: 85.31% | Val Acc: 90.81%
Epoch 2/10 | Train Acc: 91.74% | Val Acc: 94.99%
Epoch 3/10 | Train Acc: 93.30% | Val Acc: 93.46%
Epoch 4/10 | Train Acc: 96.65% | Val Acc: 97.45%
Epoch 5/10 | Train Acc: 96.90% | Val Acc: 97.22%
Epoch 6/10 | Train Acc: 96.83% | Val Acc: 97.82%
Epoch 7/10 | Train Acc: 98.18% | Val Acc: 98.37%
Epoch 8/10 | Train Acc: 98.41% | Val Acc: 98.66%
Epoch 9/10 | Train Acc: 98.33% | Val Acc: 98.60%
Epoch 10/10 | Train Acc: 98.96% | Val Acc: 99.00%
✅ Training complete!
✅ Model saved to Kaggle output!


In [5]:

# Save to Kaggle's persistent output folder
torch.save(model.state_dict(), '/kaggle/working/plant_disease_model.pth')

with open('/kaggle/working/class_names.json', 'w') as f:
    json.dump(class_names, f)

print("✅ Model saved to Kaggle output!")

✅ Model saved to Kaggle output!
